# Ingeniería de Características

**Universidad de los Andes — Minería de Datos**  
**Semana 7 — Ingeniería de Características**

---

Este notebook cubre las técnicas fundamentales de **ingeniería de características** (*feature engineering*), una de las etapas más influyentes en el desempeño de cualquier modelo de aprendizaje automático. Trabajaremos con el dataset `insurance.csv` y conjuntos de datos de scikit-learn para ilustrar cada concepto.

**Contenidos:**

1. ¿Por qué importa la ingeniería de características?
2. Transformaciones numéricas
3. Codificaciones categóricas
4. Imputación de valores faltantes
5. Pipelines con scikit-learn
6. Selección de características
7. Manejo del desbalanceo de clases

In [ ]:
# ── Importaciones globales ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix

# Selección de características
from sklearn.feature_selection import SelectKBest, chi2, f_classif, RFE

# Datasets de ejemplo
from sklearn.datasets import load_breast_cancer, make_classification

# Manejo de desbalanceo
try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("imbalanced-learn no está instalado. Instálalo con: pip install imbalanced-learn")
    IMBLEARN_AVAILABLE = False

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print("Librerías importadas correctamente.")

---

## 1. ¿Por qué importa la Ingeniería de Características?

La **ingeniería de características** es el proceso de usar conocimiento del dominio para transformar los datos crudos en representaciones que los algoritmos de ML puedan aprovechar mejor.

> *"Coming up with features is difficult, time-consuming, requires expert knowledge.  
> Applied machine learning is basically feature engineering."*  
> — Andrew Ng

El impacto práctico se puede demostrar comparando un modelo entrenado con datos crudos versus datos bien transformados.

In [ ]:
# ── Cargar el dataset de seguros ────────────────────────────────────────────
df = pd.read_csv('../insurance.csv')

print(f"Dimensiones: {df.shape}")
print(f"\nTipos de datos:")
print(df.dtypes)
df.head()

In [ ]:
# ── Demostración del impacto: modelo sin vs con transformación ──────────────

# Prepararmos una versión mínima del dataset: solo columnas numéricas
df_num = df[['age', 'bmi', 'children', 'charges']].copy()

X = df_num[['age', 'bmi', 'children']]
y = df_num['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo 1: sin transformar
modelo_raw = LinearRegression()
modelo_raw.fit(X_train, y_train)
pred_raw = modelo_raw.predict(X_test)
r2_raw = r2_score(y_test, pred_raw)

# Modelo 2: con escalado estándar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

modelo_scaled = LinearRegression()
modelo_scaled.fit(X_train_scaled, y_train)
pred_scaled = modelo_scaled.predict(X_test_scaled)
r2_scaled = r2_score(y_test, pred_scaled)

# Para regresión lineal el R² es igual (escalado no cambia el modelo lineal),
# pero la diferencia se nota en modelos sensibles a la escala (e.g. KNN, SVM)
print(f"R² sin transformar  : {r2_raw:.4f}")
print(f"R² con StandardScaler: {r2_scaled:.4f}")
print("\n→ Para regresión lineal el resultado es idéntico, pero veremos que")
print("  en modelos basados en distancias (KNN, SVM) el escalado es crucial.")

---

## 2. Transformaciones Numéricas

### 2.1 Escaladores: StandardScaler, MinMaxScaler, RobustScaler

| Escalador | Fórmula | Útil cuando… |
|---|---|---|
| **StandardScaler** | $(x - \mu) / \sigma$ | Datos aproximadamente normales |
| **MinMaxScaler** | $(x - x_{min}) / (x_{max} - x_{min})$ | Necesitamos rango $[0,1]$ |
| **RobustScaler** | $(x - Q_2) / (Q_3 - Q_1)$ | Datos con valores atípicos (outliers) |

In [ ]:
# ── Comparación visual de los tres escaladores sobre 'charges' ──────────────

charges = df[['charges']].copy()

# Aplicar cada escalador
std_scaler    = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

charges_std    = std_scaler.fit_transform(charges)
charges_minmax = minmax_scaler.fit_transform(charges)
charges_robust = robust_scaler.fit_transform(charges)

# Construir DataFrame de resultados
df_scaled = pd.DataFrame({
    'Original': charges['charges'].values,
    'StandardScaler': charges_std.ravel(),
    'MinMaxScaler': charges_minmax.ravel(),
    'RobustScaler': charges_robust.ravel()
})

# Visualizar distribuciones
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cols = ['Original', 'StandardScaler', 'MinMaxScaler', 'RobustScaler']

for ax, col in zip(axes, cols):
    ax.hist(df_scaled[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Valor')

plt.suptitle('Comparación de Escaladores — Variable: charges', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Estadísticas resumen
print("\nEstadísticas descriptivas después de escalar:")
print(df_scaled.describe().round(3))

### 2.2 Transformación Logarítmica para Datos Asimétricos

Muchas variables financieras y de seguros presentan distribuciones con **sesgo positivo** (*right-skewed*). La transformación $\log(x+1)$ reduce ese sesgo y ayuda a modelos que asumen normalidad (regresión lineal, LDA, etc.).

In [ ]:
# ── Transformación logarítmica ───────────────────────────────────────────────

# Variable objetivo: 'charges' tiene fuerte sesgo positivo
charges_log = np.log1p(df['charges'])  # log(x+1) para evitar log(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución original
axes[0].hist(df['charges'], bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Charges — Original\nSkewness: {df["charges"].skew():.2f}')
axes[0].set_xlabel('USD')

# Distribución transformada
axes[1].hist(charges_log, bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[1].set_title(f'Charges — log(1+x)\nSkewness: {charges_log.skew():.2f}')
axes[1].set_xlabel('log(USD + 1)')

plt.suptitle('Efecto de la Transformación Logarítmica', fontsize=13)
plt.tight_layout()
plt.show()

# Comparar R² con y sin transformación logarítmica en la variable objetivo
X_all = pd.get_dummies(df.drop('charges', axis=1), drop_first=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, df['charges'], test_size=0.2, random_state=42)
_, _, y_tr_log, y_te_log = train_test_split(X_all, charges_log, test_size=0.2, random_state=42)

lr = LinearRegression()

# Sin transformar
lr.fit(X_tr, y_tr)
r2_orig = r2_score(y_te, lr.predict(X_te))

# Con log en y
lr.fit(X_tr, y_tr_log)
pred_log = lr.predict(X_te)
# Revertir la transformación para comparar en la escala original
pred_original_scale = np.expm1(pred_log)
r2_log = r2_score(y_te, pred_original_scale)

print(f"R² sin transformación log en y : {r2_orig:.4f}")
print(f"R² con transformación log en y : {r2_log:.4f}")

### 2.3 Características Polinómicas

Las **características polinómicas** permiten capturar relaciones no lineales entre variables sin cambiar el modelo lineal. Para grado 2 con variables $a$ y $b$, se generan: $1, a, b, a^2, ab, b^2$.

In [ ]:
# ── Características polinómicas ─────────────────────────────────────────────

X_num = df[['age', 'bmi']].copy()

# Transformación de grado 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_num)

# Nombres de las nuevas características
nombres = poly.get_feature_names_out(['age', 'bmi'])
df_poly = pd.DataFrame(X_poly, columns=nombres)

print(f"Características originales : {X_num.shape[1]}")
print(f"Características polinómicas: {X_poly.shape[1]}")
print(f"\nNombres de columnas generadas: {list(nombres)}")
df_poly.head()

In [ ]:
# ── Comparar regresión lineal vs polinómica (grado 2 y 3) ───────────────────

X_base = df[['age', 'bmi', 'children']]
y_base = df['charges']

resultados = {}

for grado in [1, 2, 3]:
    poly_temp = PolynomialFeatures(degree=grado, include_bias=False)
    X_p = poly_temp.fit_transform(X_base)
    X_tr, X_te, y_tr, y_te = train_test_split(X_p, y_base, test_size=0.2, random_state=42)
    
    scaler_temp = StandardScaler()
    X_tr_s = scaler_temp.fit_transform(X_tr)
    X_te_s  = scaler_temp.transform(X_te)
    
    lr_temp = LinearRegression()
    lr_temp.fit(X_tr_s, y_tr)
    
    r2 = r2_score(y_te, lr_temp.predict(X_te_s))
    resultados[f'Grado {grado}'] = {'Características': X_p.shape[1], 'R²': round(r2, 4)}

print(pd.DataFrame(resultados).T.to_string())

---

## 3. Codificaciones Categóricas

Los algoritmos de ML requieren entradas numéricas. Las variables categóricas deben ser **codificadas** antes de ser usadas. La elección del método importa mucho.

### 3.1 LabelEncoder — y sus problemas

In [ ]:
# ── LabelEncoder ────────────────────────────────────────────────────────────

# LabelEncoder asigna un entero a cada categoría
le = LabelEncoder()

df_enc = df.copy()

# Codificar la columna 'region'
df_enc['region_label'] = le.fit_transform(df['region'])

print("Mapeo región → entero:")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {cls:20s} → {idx}")

print("\n⚠ PROBLEMA: el modelo asume que 'northwest=2 > northeast=1',")
print("  introduciendo un orden artificial que no existe.")
print("  → Solo usar LabelEncoder para la variable OBJETIVO o variables ORDINALES.")

### 3.2 OneHotEncoder — Variables nominales

In [ ]:
# ── OneHotEncoder ────────────────────────────────────────────────────────────

# OneHotEncoder crea una columna binaria por cada categoría
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' evita multicolinealidad

region_ohe = ohe.fit_transform(df[['region']])
columnas_ohe = ohe.get_feature_names_out(['region'])

df_ohe = pd.DataFrame(region_ohe, columns=columnas_ohe)

print("Columnas generadas por OneHotEncoder:")
print(columnas_ohe)
print()
print(df_ohe.head(8).to_string(index=False))

### 3.3 OrdinalEncoder — Variables con orden natural

In [ ]:
# ── OrdinalEncoder ───────────────────────────────────────────────────────────

# Creamos un ejemplo con una variable ordinal: nivel de riesgo
df_riesgo = pd.DataFrame({
    'nivel_riesgo': ['Bajo', 'Alto', 'Medio', 'Bajo', 'Alto', 'Medio', 'Alto', 'Bajo']
})

# Definimos el orden explícitamente: Bajo < Medio < Alto
oe = OrdinalEncoder(categories=[['Bajo', 'Medio', 'Alto']])
df_riesgo['nivel_codificado'] = oe.fit_transform(df_riesgo[['nivel_riesgo']])

print("Codificación ordinal (Bajo=0, Medio=1, Alto=2):")
print(df_riesgo.to_string(index=False))

# Aplicar al dataset de seguros: 'smoker' es binario (yes/no)
oe_smoker = OrdinalEncoder(categories=[['no', 'yes']])
df_enc['smoker_ord'] = oe_smoker.fit_transform(df[['smoker']])
print("\nPrimeras filas con smoker codificado:")
print(df_enc[['smoker', 'smoker_ord']].drop_duplicates().head())

---

## 4. Imputación de Valores Faltantes

En la realidad, los datos casi siempre tienen **valores faltantes** (NA, NaN). Eliminar las filas con faltantes puede sesgar el análisis o perder información valiosa. La **imputación** reemplaza los faltantes con estimaciones.

Estrategias:
- **Media / Mediana / Moda**: simple, rápido, pero no considera la estructura de los datos
- **KNN Imputer**: usa los $k$ vecinos más cercanos para estimar el valor faltante
- **IterativeImputer**: modela cada variable con faltantes como función de las demás (más avanzado)

In [ ]:
# ── Crear valores faltantes artificiales para demostración ──────────────────
np.random.seed(42)

df_missing = df[['age', 'bmi', 'children', 'charges']].copy()

# Introducir ~10% de valores faltantes en 'bmi' y 'age'
n_missing_bmi = int(0.10 * len(df_missing))
n_missing_age = int(0.08 * len(df_missing))

idx_bmi = np.random.choice(df_missing.index, size=n_missing_bmi, replace=False)
idx_age = np.random.choice(df_missing.index, size=n_missing_age, replace=False)

df_missing.loc[idx_bmi, 'bmi'] = np.nan
df_missing.loc[idx_age, 'age'] = np.nan

print("Valores faltantes introducidos:")
print(df_missing.isnull().sum())
print(f"\nTotal de celdas: {df_missing.size}")
print(f"Porcentaje faltante: {df_missing.isnull().mean().mean()*100:.1f}%")

In [ ]:
# ── SimpleImputer: media, mediana y moda ────────────────────────────────────

estrategias = ['mean', 'median', 'most_frequent']
resultados_imp = {}

for estrategia in estrategias:
    imp = SimpleImputer(strategy=estrategia)
    df_imp = pd.DataFrame(
        imp.fit_transform(df_missing),
        columns=df_missing.columns
    )
    # Guardar los valores imputados en 'bmi' para compararlos
    bmi_imputados = df_imp.loc[idx_bmi, 'bmi']
    resultados_imp[estrategia] = bmi_imputados.mean()

print("Valor medio imputado en 'bmi' por estrategia:")
for k, v in resultados_imp.items():
    print(f"  {k:15s}: {v:.4f}")

print(f"\nValor real promedio de 'bmi' (sin NaN): {df['bmi'].mean():.4f}")

In [ ]:
# ── KNNImputer ───────────────────────────────────────────────────────────────

# KNNImputer usa los k vecinos más cercanos para estimar el valor faltante
knn_imp = KNNImputer(n_neighbors=5)
df_knn = pd.DataFrame(
    knn_imp.fit_transform(df_missing),
    columns=df_missing.columns
)

# Comparar distribuciones: original vs imputada
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# BMI
axes[0].hist(df['bmi'], bins=30, alpha=0.6, label='Original completo', color='steelblue')
axes[0].hist(df_knn['bmi'], bins=30, alpha=0.6, label='KNN imputado', color='coral')
axes[0].set_title('Distribución de BMI: Original vs KNN Imputado')
axes[0].set_xlabel('BMI')
axes[0].legend()

# Age
axes[1].hist(df['age'], bins=30, alpha=0.6, label='Original completo', color='steelblue')
axes[1].hist(df_knn['age'], bins=30, alpha=0.6, label='KNN imputado', color='orange')
axes[1].set_title('Distribución de Age: Original vs KNN Imputado')
axes[1].set_xlabel('Edad')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Las distribuciones se preservan bien con KNNImputer.")

In [ ]:
# ── Comparación cuantitativa de imputadores ──────────────────────────────────

# Calculamos el ECM (error cuadrático medio) respecto a los valores reales
valores_reales_bmi = df.loc[idx_bmi, 'bmi'].values

comparacion = {}

for estrategia in ['mean', 'median']:
    imp_temp = SimpleImputer(strategy=estrategia)
    df_temp = pd.DataFrame(imp_temp.fit_transform(df_missing), columns=df_missing.columns)
    ecm = mean_squared_error(valores_reales_bmi, df_temp.loc[idx_bmi, 'bmi'].values)
    comparacion[f'SimpleImputer ({estrategia})'] = round(ecm, 4)

# KNN
ecm_knn = mean_squared_error(valores_reales_bmi, df_knn.loc[idx_bmi, 'bmi'].values)
comparacion['KNNImputer (k=5)'] = round(ecm_knn, 4)

print("ECM de la imputación de 'bmi' (menor = mejor):")
for k, v in sorted(comparacion.items(), key=lambda x: x[1]):
    print(f"  {k:30s}: {v}")

---

## 5. Pipelines con scikit-learn

Un **Pipeline** encadena múltiples pasos de transformación y un estimador final en un objeto único. Esto garantiza:

1. **Reproducibilidad**: los mismos pasos se aplican en entrenamiento y predicción
2. **Prevención de fuga de datos** (*data leakage*): el `fit` del escalador usa solo datos de entrenamiento
3. **Mantenibilidad**: el código es más limpio y fácil de depurar

### 5.1 ColumnTransformer — Transformaciones por tipo de variable

In [ ]:
# ── ColumnTransformer: aplica distintas transformaciones a distintas columnas ─

# Separar columnas por tipo
columnas_numericas     = ['age', 'bmi', 'children']
columnas_categoricas   = ['sex', 'smoker', 'region']

# Definir transformador para columnas numéricas: imputación + escalado
transformador_num = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # primero imputar
    ('scaler', StandardScaler())                     # luego escalar
])

# Definir transformador para columnas categóricas: imputación + OHE
transformador_cat = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # imputar con moda
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))  # OHE
])

# Combinar en ColumnTransformer
preprocesador = ColumnTransformer(transformers=[
    ('num', transformador_num, columnas_numericas),
    ('cat', transformador_cat, columnas_categoricas)
])

print("ColumnTransformer definido:")
print(preprocesador)

### 5.2 Pipeline completo: preprocesamiento + modelo

In [ ]:
# ── Pipeline end-to-end con el dataset de seguros ───────────────────────────

# Variables de entrada y objetivo
X = df.drop('charges', axis=1)
y = df['charges']

# División entrenamiento / prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Construir pipeline completo: preprocesamiento + regresión lineal
pipeline_lr = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', LinearRegression())
])

# Entrenar
pipeline_lr.fit(X_train, y_train)

# Evaluar
y_pred = pipeline_lr.predict(X_test)
r2  = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.2f} USD")

In [ ]:
# ── Visualizar predicciones vs valores reales ────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: real vs predicho
axes[0].scatter(y_test, y_pred, alpha=0.4, color='steelblue', edgecolors='white', s=40)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valor Real (USD)')
axes[0].set_ylabel('Valor Predicho (USD)')
axes[0].set_title(f'Real vs Predicho — R²={r2:.3f}')
axes[0].legend()

# Residuos
residuos = y_test - y_pred
axes[1].hist(residuos, bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residuo (USD)')
axes[1].set_title('Distribución de Residuos')

plt.suptitle('Evaluación del Pipeline de Regresión', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparar múltiples modelos usando el mismo pipeline ──────────────────────

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

modelos = {
    'Regresión Lineal': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5)
}

resultados_modelos = []

for nombre, modelo in modelos.items():
    # Construir pipeline con el mismo preprocesador pero diferente modelo
    pipe = Pipeline(steps=[
        ('preprocesamiento', preprocesador),
        ('modelo', modelo)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    resultados_modelos.append({
        'Modelo': nombre,
        'R²': round(r2_score(y_test, preds), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 2)
    })

df_resultados = pd.DataFrame(resultados_modelos).sort_values('R²', ascending=False)
print(df_resultados.to_string(index=False))

---

## 6. Selección de Características

Con conjuntos de datos de alta dimensionalidad, no todas las variables son útiles. Las características irrelevantes o redundantes pueden:
- Incrementar el tiempo de entrenamiento
- Causar sobreajuste (*overfitting*)
- Dificultar la interpretación

Existen tres familias de métodos:

| Método | Descripción | Ventaja |
|---|---|---|
| **Filter** | Criterio estadístico, independiente del modelo | Rápido, escalable |
| **Wrapper** | Entrena modelos con distintos subconjuntos | Alta precisión |
| **Embedded** | La selección ocurre durante el entrenamiento | Eficiente |

Usaremos el dataset de **cáncer de mama** de scikit-learn (30 características, clasificación binaria).

In [ ]:
# ── Cargar dataset breast_cancer ─────────────────────────────────────────────

bc = load_breast_cancer()
X_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
y_bc = bc.target  # 0 = maligno, 1 = benigno

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

print(f"Dataset: {X_bc.shape[0]} muestras, {X_bc.shape[1]} características")
print(f"Clases: {np.bincount(y_bc)}  (0=maligno, 1=benigno)")

### 6.1 Métodos de Filtro: SelectKBest

In [ ]:
# ── SelectKBest con f_classif (ANOVA F-statistic) ───────────────────────────

# Escalar primero (necesario para f_classif)
scaler_bc = StandardScaler()
X_bc_scaled = scaler_bc.fit_transform(X_bc_train)

# Seleccionar las 10 mejores características
selector_kbest = SelectKBest(score_func=f_classif, k=10)
selector_kbest.fit(X_bc_scaled, y_bc_train)

# Obtener puntuaciones
df_scores = pd.DataFrame({
    'Característica': bc.feature_names,
    'F-Score': selector_kbest.scores_,
    'p-valor': selector_kbest.pvalues_,
    'Seleccionada': selector_kbest.get_support()
}).sort_values('F-Score', ascending=False)

# Visualizar
fig, ax = plt.subplots(figsize=(12, 5))
colores = ['steelblue' if s else 'lightgray' for s in df_scores['Seleccionada']]
ax.barh(df_scores['Característica'], df_scores['F-Score'], color=colores)
ax.set_xlabel('F-Score')
ax.set_title('SelectKBest (f_classif) — Top 10 características resaltadas')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 características seleccionadas:")
print(df_scores[df_scores['Seleccionada']]['Característica'].values)

In [ ]:
# ── Evaluar impacto de selección de características en el modelo ─────────────

X_bc_test_scaled = scaler_bc.transform(X_bc_test)

# Con todas las características
lr_bc = LogisticRegression(max_iter=1000, random_state=42)
lr_bc.fit(X_bc_scaled, y_bc_train)
acc_all = lr_bc.score(X_bc_test_scaled, y_bc_test)

# Con top-10 características
X_bc_train_k10 = selector_kbest.transform(X_bc_scaled)
X_bc_test_k10  = selector_kbest.transform(X_bc_test_scaled)

lr_k10 = LogisticRegression(max_iter=1000, random_state=42)
lr_k10.fit(X_bc_train_k10, y_bc_train)
acc_k10 = lr_k10.score(X_bc_test_k10, y_bc_test)

print(f"Accuracy con 30 características : {acc_all:.4f}")
print(f"Accuracy con 10 características : {acc_k10:.4f}")
print(f"\n→ Se reduce el número de variables a {10/30*100:.0f}% conservando casi todo el rendimiento.")

### 6.2 Métodos Wrapper: RFE (Recursive Feature Elimination)

In [ ]:
# ── RFE con Regresión Logística ──────────────────────────────────────────────

# RFE entrena el modelo repetidamente eliminando las características menos importantes
lr_rfe = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=lr_rfe, n_features_to_select=10, step=1)
rfe.fit(X_bc_scaled, y_bc_train)

# Ranking de características (1 = seleccionada)
df_rfe = pd.DataFrame({
    'Característica': bc.feature_names,
    'Ranking': rfe.ranking_,
    'Seleccionada': rfe.support_
}).sort_values('Ranking')

print("Top 10 características seleccionadas por RFE:")
print(df_rfe[df_rfe['Seleccionada']]['Característica'].values)

# Accuracy con RFE
X_bc_train_rfe = rfe.transform(X_bc_scaled)
X_bc_test_rfe  = rfe.transform(X_bc_test_scaled)

lr_final = LogisticRegression(max_iter=1000, random_state=42)
lr_final.fit(X_bc_train_rfe, y_bc_train)
acc_rfe = lr_final.score(X_bc_test_rfe, y_bc_test)

print(f"\nAccuracy con RFE (10 vars): {acc_rfe:.4f}")

### 6.3 Métodos Embedded: Importancia de características en árboles

In [ ]:
# ── Feature Importances con RandomForest ────────────────────────────────────

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_bc_train, y_bc_train)  # Random Forest no requiere escalado

# Importancias
df_imp = pd.DataFrame({
    'Característica': bc.feature_names,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=False)

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))
umbral = df_imp['Importancia'].nlargest(10).min()  # umbral top-10
colores = ['steelblue' if v >= umbral else 'lightgray' for v in df_imp['Importancia']]
ax.barh(df_imp['Característica'], df_imp['Importancia'], color=colores)
ax.axvline(umbral, color='red', linestyle='--', alpha=0.7, label='Umbral top-10')
ax.set_xlabel('Importancia (Gini)')
ax.set_title('Importancia de Características — Random Forest')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

print("Top 10 características más importantes:")
print(df_imp.head(10)['Característica'].values)

In [ ]:
# ── Comparación de los tres métodos de selección ────────────────────────────

# Selección por importancia de RF: top 10
top10_rf = df_imp.head(10)['Característica'].values
X_bc_train_rf = X_bc_train[top10_rf]
X_bc_test_rf  = X_bc_test[top10_rf]

scaler_rf = StandardScaler()
X_bc_train_rf_s = scaler_rf.fit_transform(X_bc_train_rf)
X_bc_test_rf_s  = scaler_rf.transform(X_bc_test_rf)

lr_emb = LogisticRegression(max_iter=1000, random_state=42)
lr_emb.fit(X_bc_train_rf_s, y_bc_train)
acc_emb = lr_emb.score(X_bc_test_rf_s, y_bc_test)

print("Comparación de métodos de selección de características (top 10 de 30):")
print(f"  {'Todas las características (30)':35s}: {acc_all:.4f}")
print(f"  {'SelectKBest (f_classif)':35s}: {acc_k10:.4f}")
print(f"  {'RFE (LogisticRegression)':35s}: {acc_rfe:.4f}")
print(f"  {'RandomForest importances':35s}: {acc_emb:.4f}")

---

## 7. Manejo del Desbalanceo de Clases

El **desbalanceo de clases** ocurre cuando una clase tiene significativamente más muestras que otra (e.g., detección de fraude: 99% legítimo, 1% fraude). Esto puede hacer que el modelo simplemente prediga siempre la clase mayoritaria.

Estrategias principales:

| Técnica | Tipo | Descripción |
|---|---|---|
| `class_weight='balanced'` | Ajuste del modelo | Penaliza más los errores en la clase minoritaria |
| `RandomOverSampler` | Sobremuestreo | Replica muestras de la clase minoritaria |
| `RandomUnderSampler` | Submuestreo | Elimina muestras de la clase mayoritaria |
| `SMOTE` | Sobremuestreo sintético | Genera nuevas muestras interpolando entre vecinos |

### 7.1 Crear un dataset desbalanceado

In [ ]:
# ── Generar dataset desbalanceado sintético ──────────────────────────────────

X_imb, y_imb = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=6,
    weights=[0.90, 0.10],  # 90% clase 0, 10% clase 1
    random_state=42
)

print("Distribución de clases en el dataset desbalanceado:")
unique, counts = np.unique(y_imb, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Clase {cls}: {cnt} muestras ({cnt/len(y_imb)*100:.1f}%)")

# División
X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

### 7.2 class_weight en scikit-learn

In [ ]:
# ── Comparar sin y con class_weight='balanced' ───────────────────────────────

from sklearn.metrics import f1_score, recall_score, precision_score

def evaluar_modelo(nombre, y_true, y_pred):
    """Imprime métricas de clasificación para la clase minoritaria (1)."""
    print(f"\n{'─'*45}")
    print(f"Modelo: {nombre}")
    print(f"  Accuracy  : {(y_true == y_pred).mean():.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred):.4f}")

# Modelo sin ajuste de pesos
lr_nopeso = LogisticRegression(max_iter=500, random_state=42)
lr_nopeso.fit(X_imb_train, y_imb_train)
evaluar_modelo('Sin class_weight', y_imb_test, lr_nopeso.predict(X_imb_test))

# Modelo con class_weight='balanced'
lr_balanced = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
lr_balanced.fit(X_imb_train, y_imb_train)
evaluar_modelo('Con class_weight=balanced', y_imb_test, lr_balanced.predict(X_imb_test))

### 7.3 RandomOverSampler y RandomUnderSampler

In [ ]:
# ── Sobremuestreo y submuestreo aleatorio ────────────────────────────────────

if IMBLEARN_AVAILABLE:
    # Sobremuestreo: replica muestras de la clase minoritaria
    ros = RandomOverSampler(random_state=42)
    X_over, y_over = ros.fit_resample(X_imb_train, y_imb_train)

    # Submuestreo: elimina muestras de la clase mayoritaria
    rus = RandomUnderSampler(random_state=42)
    X_under, y_under = rus.fit_resample(X_imb_train, y_imb_train)

    print("Distribución de clases después del remuestreo:")
    for nombre, y_r in [('Original (train)', y_imb_train),
                         ('Sobremuestreo', y_over),
                         ('Submuestreo', y_under)]:
        cnt = np.bincount(y_r)
        print(f"  {nombre:25s}: Clase 0={cnt[0]}, Clase 1={cnt[1]}, Total={sum(cnt)}")

    # Entrenar modelos con datos remuestreados
    for nombre_res, X_r, y_r in [('RandomOverSampler', X_over, y_over),
                                   ('RandomUnderSampler', X_under, y_under)]:
        lr_r = LogisticRegression(max_iter=500, random_state=42)
        lr_r.fit(X_r, y_r)
        evaluar_modelo(nombre_res, y_imb_test, lr_r.predict(X_imb_test))
else:
    print("Instala imbalanced-learn para ejecutar esta celda: pip install imbalanced-learn")

### 7.4 SMOTE — Synthetic Minority Oversampling Technique

SMOTE no simplemente replica muestras existentes, sino que **genera muestras sintéticas** interpolando entre puntos vecinos de la clase minoritaria. Esto produce mayor variabilidad que el sobremuestreo aleatorio.

$$x_{nuevo} = x_i + \lambda \cdot (x_{vecino} - x_i), \quad \lambda \sim U(0,1)$$

In [ ]:
# ── SMOTE ────────────────────────────────────────────────────────────────────

if IMBLEARN_AVAILABLE:
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_smote, y_smote = smote.fit_resample(X_imb_train, y_imb_train)

    print(f"Antes de SMOTE : {np.bincount(y_imb_train)}")
    print(f"Después de SMOTE: {np.bincount(y_smote)}")

    # Modelo entrenado con SMOTE
    lr_smote = LogisticRegression(max_iter=500, random_state=42)
    lr_smote.fit(X_smote, y_smote)
    evaluar_modelo('SMOTE', y_imb_test, lr_smote.predict(X_imb_test))
else:
    print("Instala imbalanced-learn: pip install imbalanced-learn")

In [ ]:
# ── Comparación visual de todas las estrategias ──────────────────────────────

if IMBLEARN_AVAILABLE:
    estrategias_desbal = {
        'Sin ajuste': lr_nopeso,
        'class_weight=balanced': lr_balanced,
    }

    # Agregar modelos de remuestreo
    for nombre_res, X_r, y_r in [('RandomOverSampler', X_over, y_over),
                                   ('RandomUnderSampler', X_under, y_under),
                                   ('SMOTE', X_smote, y_smote)]:
        lr_temp = LogisticRegression(max_iter=500, random_state=42)
        lr_temp.fit(X_r, y_r)
        estrategias_desbal[nombre_res] = lr_temp

    metricas = []
    for nombre_e, modelo_e in estrategias_desbal.items():
        pred_e = modelo_e.predict(X_imb_test)
        metricas.append({
            'Estrategia': nombre_e,
            'Accuracy': (y_imb_test == pred_e).mean(),
            'F1 (clase 1)': f1_score(y_imb_test, pred_e),
            'Recall (clase 1)': recall_score(y_imb_test, pred_e)
        })

    df_metricas = pd.DataFrame(metricas).set_index('Estrategia')

    # Gráfica de barras agrupadas
    df_metricas.plot(kind='bar', figsize=(12, 5), rot=30, colormap='Set2', edgecolor='white')
    plt.title('Comparación de Estrategias para Manejo del Desbalanceo')
    plt.ylabel('Métrica')
    plt.ylim(0, 1.05)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

    print("\nResumen numérico:")
    print(df_metricas.round(4).to_string())
else:
    print("Instala imbalanced-learn: pip install imbalanced-learn")

---

## 8. Resumen y Buenas Prácticas

### Checklist de Ingeniería de Características

**Transformaciones numéricas:**
- Usar `StandardScaler` para modelos que asumen normalidad (LR, SVM, KNN)
- Usar `RobustScaler` cuando hay outliers significativos
- Aplicar transformación log cuando la variable tiene sesgo positivo fuerte
- Las características polinómicas aumentan la capacidad del modelo pero pueden causar sobreajuste

**Codificación:**
- Nunca usar `LabelEncoder` para variables nominales de entrada (solo para la variable objetivo)
- Usar `OneHotEncoder` con `drop='first'` para evitar multicolinealidad perfecta
- Usar `OrdinalEncoder` solo cuando el orden es real y conocido

**Imputación:**
- Siempre ajustar (`fit`) el imputador **solo** con datos de entrenamiento
- `KNNImputer` da mejores resultados pero es más costoso computacionalmente

**Pipelines:**
- Encapsular siempre el preprocesamiento en un Pipeline para evitar data leakage
- Usar `ColumnTransformer` para aplicar transformaciones distintas por tipo de columna

**Selección de características:**
- Empezar con métodos de filtro (rápidos), luego refinar con wrapper o embedded
- Validar con cross-validation que la selección realmente mejora el rendimiento

**Desbalanceo:**
- Evaluar con F1, Recall, AUC-ROC, no solo Accuracy
- SMOTE solo sobre el conjunto de entrenamiento, nunca sobre el de prueba
- `class_weight='balanced'` es la opción más sencilla y casi siempre competitiva

In [ ]:
# ── Pipeline final end-to-end: insurance dataset completo ───────────────────
# Este pipeline incluye todos los pasos aprendidos en el notebook

# Reutilizamos el preprocesador definido en la sección 5
X_final = df.drop('charges', axis=1)
y_final = df['charges']

X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42
)

# Pipeline con Gradient Boosting (el mejor modelo de la sección 5)
from sklearn.ensemble import GradientBoostingRegressor

pipeline_final = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ))
])

pipeline_final.fit(X_tr_f, y_tr_f)

y_pred_f = pipeline_final.predict(X_te_f)
r2_f  = r2_score(y_te_f, y_pred_f)
rmse_f = np.sqrt(mean_squared_error(y_te_f, y_pred_f))

print("Pipeline final — Gradient Boosting con preprocesamiento completo:")
print(f"  R²  : {r2_f:.4f}")
print(f"  RMSE: {rmse_f:.2f} USD")

# Ejemplo de predicción con nuevos datos (sin necesidad de preprocesar manualmente)
nuevo_paciente = pd.DataFrame({
    'age': [35], 'sex': ['male'], 'bmi': [27.5],
    'children': [2], 'smoker': ['no'], 'region': ['southwest']
})

prediccion = pipeline_final.predict(nuevo_paciente)
print(f"\nPredicción de charges para el nuevo paciente: ${prediccion[0]:,.2f} USD")

---

## Ejercicios Propuestos

1. **Transformaciones**: Aplica `PowerTransformer` (Box-Cox o Yeo-Johnson) sobre `charges` y compara la reducción de sesgo con la transformación log. ¿Cuál es mejor?

2. **Codificación**: Investiga `TargetEncoder` de scikit-learn. ¿En qué situaciones es preferible sobre `OneHotEncoder`? Aplícalo a la columna `region`.

3. **Pipeline completo**: Construye un pipeline para clasificar si un paciente es fumador (`smoker`) basándote en las demás variables. Incluye preprocesamiento, selección de características y un modelo de clasificación.

4. **Desbalanceo**: Crea un dataset desbalanceado a partir del dataset de seguros (clase 1: `charges > 30000`, clase 0: resto). Aplica SMOTE y compara el F1-score con y sin remuestreo usando Random Forest.

5. **Selección de características**: Usa `RFECV` (RFE con validación cruzada) sobre el dataset breast_cancer para encontrar automáticamente el número óptimo de características. Grafica el score de validación cruzada vs número de características.

## Referencias

**Texto guía**
- Notas de clase del curso (disponibles en el repositorio).

**Bibliografía complementaria**
- García, S., Luengo, J., & Herrera, F. (2015). *Data Preprocessing in Data Mining* (Cap. 3). Springer.
- Kuhn, M., & Johnson, K. (2019). *Feature Engineering and Selection: A Practical Approach for Predictive Models*. CRC Press.
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed., Cap. 2). O'Reilly Media.
- Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *JAIR*, 16, 321–357.